# Notebook 3: Multi-Head Latent Attention (MLA) - JAX

In [ ]:
import jax.numpy as jnp
from flax import linen as nn
from jax import random

class MultiHeadLatentAttention(nn.Module):
    """MLA in Flax"""
    hidden_size: int
    num_heads: int
    num_kv_heads: int = 1
    
    def setup(self):
        self.head_dim = self.hidden_size // self.num_heads
        self.latent_dim = self.num_kv_heads * self.head_dim // 2
        
        self.W_q = nn.Dense(self.hidden_size)
        self.W_KV = nn.Dense(self.latent_dim * 2)  # Compressed
        self.W_o = nn.Dense(self.hidden_size)
    
    def __call__(self, x, causal_mask=True):
        batch, seq_len, _ = x.shape
        
        # Q projection
        Q = self.W_q(x).reshape(batch, seq_len, self.num_heads, self.head_dim)
        Q = jnp.transpose(Q, (0, 2, 1, 3))
        
        # KV projection (compressed)
        KV = self.W_KV(x)
        K_latent = KV[..., :self.latent_dim]
        V_latent = KV[..., self.latent_dim:]
        
        # Expand to heads
        K = jnp.broadcast_to(K_latent[:, None, :, :], (batch, self.num_kv_heads, seq_len, self.head_dim))
        V = jnp.broadcast_to(V_latent[:, None, :, :], (batch, self.num_kv_heads, seq_len, self.head_dim))
        
        # GQA expansion
        if self.num_kv_heads < self.num_heads:
            repeat = self.num_heads // self.num_kv_heads
            K = jnp.repeat(K, repeat, axis=1)
            V = jnp.repeat(V, repeat, axis=1)
        
        # Attention
        scale = jnp.sqrt(self.head_dim)
        attn_scores = jnp.einsum('bhqd,bhkd->bhqk', Q, K) / scale
        
        if causal_mask:
            mask = jnp.tril(jnp.ones((seq_len, seq_len)))
            attn_scores = jnp.where(mask == 0, -1e9, attn_scores)
        
        attn_weights = nn.softmax(attn_scores, axis=-1)
        attn_output = jnp.einsum('bhqk,bhvd->bhqd', attn_weights, V)
        
        attn_output = jnp.transpose(attn_output, (0, 2, 1, 3))
        attn_output = attn_output.reshape(batch, seq_len, self.hidden_size)
        
        return self.W_o(attn_output)

mla = MultiHeadLatentAttention(hidden_size=512, num_heads=8, num_kv_heads=2)
rng = random.PRNGKey(42)
x = random.normal(rng, (2, 16, 512))
params = mla.init(rng, x)
output = mla.apply(params, x)
print(f"MLA input: {x.shape}")
print(f"MLA output: {output.shape}")

## Verification
1. ✅ Can implement MLA in Flax
2. ✅ Understand KV compression
3. ✅ Can use einsum for attention